# Smoke test for `ehtim.plotting.interactive`

Open in VSCode, pick the **jax-ehtim** kernel (top-right corner of the notebook), and run cells one at a time. Plotly figures render inline in the cell output — interactive (hover, zoom, pan, double-click to reset) without writing any HTML to disk.

In [1]:
import ehtim as eh
from ehtim.plotting import interactive

obs = eh.obsdata.load_uvfits('../data/sample.uvfits')
sites = sorted({s for pair in zip(obs.data['t1'], obs.data['t2']) for s in pair})
print('sites in sample.uvfits:', sites)

Welcome to eht-imaging! v 1.3.0 

Loading uvfits:  ../data/sample.uvfits
POLREP_UVFITS: circ
Number of uvfits Correlation Products: 4
No NX table in uvfits!
sites in sample.uvfits: ['ALMA', 'LMT', 'PDB', 'PV', 'SMA', 'SMT', 'SPT']


## `plot_bl` — single baseline

Pass `site1` and `site2` for one specific baseline. No legend (only one trace).

In [2]:
interactive.plot_bl(obs, 'ALMA', 'LMT', 'amp', show=False)

## `plot_bl` — all baselines, with the Color toolbar

Omit `site1`/`site2` to plot every baseline as a uniform-gray trace. Two HTML buttons are injected above the plot:

- **Color: off → on** (single click). While on, clicking a legend entry paints that baseline from the palette; clicking it again returns it to gray. Plotly's default hide-on-click is suppressed while color mode is on.
- **Show all / reset.** Restores every managed trace to gray + visible and exits color mode.

Outside color mode, legend clicks fall through to plotly's defaults: one click hides the trace, a second click restores it. Double-click still isolates one trace (plotly default).

Use `interactive.display(fig)` instead of returning `fig` so the toolbar + click handlers are injected into the notebook output.

The toolbar is an HTML element above the chart (siblings of the plotly div), not a plotly updatemenu — that's deliberate, since plotly re-renders the layout DOM on some updates and was dropping the click listeners we attached when the toolbar lived inside the layout.


In [3]:
fig = interactive.plot_bl(obs, field='amp', show=False)
interactive.display(fig)

Try other fields — `'phase'`, `'snr'`, `'uvdist'`. The `'u'`/`'v'`/`'uvdist'` axes auto-scale to **Gλ** (or **Mλ** for shorter arrays) so labels read e.g. `4.2 Gλ` instead of `4.2e9 λ`.

SNR-cut example: `interactive.plot_bl(obs, field='amp', snrcut=5, show=False)`.


## `plotall` — scatter any two visibility fields

Plotly counterpart of `obs.plotall(field1, field2, ...)`.

- **`tag_bl=True` (default):** one trace per baseline, toggleable via legend.
- **`tag_bl=False`:** all visibilities pooled into one trace — useful when you just want the cloud.

In [4]:
# UV coverage. conj=True overlays the Hermitian conjugate half (-u,-v).
# Click a baseline in the legend to highlight it in colour; click again to hide.
fig = interactive.plotall(obs, 'u', 'v', conj=True, show=False)
interactive.display(fig)

In [5]:
# Radplot: amplitude vs baseline length. Pooled mode — single trace, useful
# when you just want to see the overall distribution.
interactive.plotall(obs, 'uvdist', 'amp', tag_bl=False, show=False)

In [6]:
# Per-baseline radplot. Click in the legend to highlight one in colour.
fig = interactive.plotall(obs, 'uvdist', 'amp', tag_bl=True, show=False)
interactive.display(fig)

## `plot_gains` — needs a Caltable

`sample.uvfits` has no caltable attached, so we synthesise a tiny fake one. For a real demo, replace this with a Caltable from `self_cal(obs, im, caltable=True)`.

`pol='both'` draws R and L as separate traces per site — R uses circles, L uses squares, so they stay separable even within the same colour.

In [7]:
import numpy as np
from ehtim.caltable import Caltable

eht = eh.array.load_txt('../arrays/EHT2017.txt')
demo_sites = ['ALMA', 'LMT', 'SMA', 'SMT']
rng = np.random.default_rng(0)
times = np.linspace(0.0, 24.0, 20)
datadict = {
    s: np.array(
        [(t, 1.0 + 0.1*rng.standard_normal() + 0j,
              1.0 + 0.1*rng.standard_normal() + 0j) for t in times],
        dtype=[('time','f8'),('rscale','c16'),('lscale','c16')]
    ) for s in demo_sites
}
ct = Caltable(ra=0.0, dec=0.0, rf=230e9, bw=4e9, datadict=datadict,
              tarr=eht.tarr, source='demo', mjd=58000, timetype='GMST')

interactive.plot_gains(ct, sites='all', gain_type='amp', pol='both', show=False)

## `dashboard` — combined view

2×2 grid: image + a data-product panel + gains + D-terms. Use this to inspect a full reconstruction (or a model + synthetic obs + self-cal output) at a glance.

**Per-panel legends.** Each scatter panel has its own legend on the right (data/model, sites, D-terms), so toggling visibility in one panel does not affect the others.

**`data_product` dropdown.** The top-of-figure dropdown switches what panel 2 shows. Options ported from `comp_plots.py`:

| option | source |
|---|---|
| Amplitude vs uv-distance | `obs.unpack(['uvdist','amp'])` + model |
| Re(vis) vs uv-distance | `obs.unpack(['vis'])` + model |
| Phase vs uv-distance | `obs.unpack(['phase'])` + model |
| χ residual vs uv-distance | `(amp − model_amp) / sigma` |
| Closure phase vs time | `obs.cphase_tri(s1,s2,s3)` + model **— all triangles overlaid** |
| Log closure amp vs time | `obs.camp_quad(..., ctype='logcamp')` + model **— all quads overlaid** |
| Closure phase vs triangle uv-area | `cphase_tri` + `obsh.uv_area_triangle` (per-triangle traces) |
| Log closure amp vs quadrangle uv-area | `camp_quad` + `obsh.uv_area_quadrangle` (per-quadrangle traces) |

For the closure-quantity products, the panel enumerates every triangle (or quadrangle) sorted by sample count, capped by `n_triangles` / `n_quadrangles` (default 12 to keep the legend readable). Each closure path gets its own palette colour and a paired model overlay (open `x` markers). Filter to a single one with `triangle=(s1,s2,s3)` or `quadrangle=(s1,s2,s3,s4)`.

**`plotp=True`** mirrors `Image.display(plotp=True)`: turns the Stokes-I colorbar on (label `Jy/px`) and overlays EVPA ticks per pixel, gated by `pcut` (I > pcut · Imax) and `mcut` (|m| > mcut). With `vec_cfun=True` the ticks are sampled by |m| via a small side colorbar.


In [8]:
# Build a (Image, Obsdata, Caltable) triple end-to-end so the dashboard has
# real data to show. Takes ~10 s — self_cal does the heavy lifting.
from ehtim.calibrating import self_cal

im_dash = eh.image.load_txt('../models/avery_sgra_eofn.txt')
eht_dash = eh.array.load_txt('../arrays/EHT2017.txt')

# Synthesise small D-terms so the D-term panel isn't all-zero.
rng = np.random.default_rng(42)
n = len(eht_dash.tarr)
eht_dash.tarr['dr'] = rng.normal(0, 0.05, n) + 1j * rng.normal(0, 0.05, n)
eht_dash.tarr['dl'] = rng.normal(0, 0.05, n) + 1j * rng.normal(0, 0.05, n)

obs_dash = im_dash.observe(eht_dash, tint=30, tadv=600,
                            tstart=0, tstop=24, bw=4e9,
                            sgrscat=False, ampcal=False, phasecal=False)
ct_dash = self_cal.self_cal(obs_dash, im_dash, processes=-1,
                             ttype='direct', caltable=True)

fig_dash = interactive.dashboard(im_dash, obs_dash, ct_dash, show=False)
interactive.display(fig_dash)

Loading text image:  ../models/avery_sgra_eofn.txt
Generating empty observation file . . . 
Producing clean visibilities from image with nfft FT . . . 
Adding gain + phase errors to data and applying a priori calibration . . . 
   Applying gain corruption: ampcal-->False
   Applying atmospheric phase corruption: phasecal-->False
Adding thermal noise to data . . . 
No stations specified in self cal: defaulting to calibrating all stations!
Computing the Model Visibilities with direct Fourier Transform...
Producing clean visibilities from image with direct FT . . . 
Not Using Multiprocessing
Scan 106/107 : [----------------------------- ]99%
self_cal time: 7.822939 s
Producing clean visibilities from image with direct FT . . . 


## Verifying the Gλ / Mλ axis auto-scaling

`plot_bl`, `plotall`, and the `dashboard` panel-2 uv-axes pick a single scale across all traces:

- max baseline ≥ 1e9 λ → display in **Gλ**
- otherwise → display in **Mλ**

Errors share the same divisor; hover tooltips and axis titles both reflect the chosen unit.


In [9]:
fig_uv = interactive.plot_bl(obs, field='uvdist', show=False)
print('y-axis title:', fig_uv.layout.yaxis.title.text)
interactive.display(fig_uv)


y-axis title: $u-v$ Distance (Gλ)


## `dashboard(plotp=True)` — polarization ticks + colorbar

Mirrors `Image.display(plotp=True)`. Turn on the Stokes-I colorbar (label `Jy/px`) and overlay EVPA tick segments at the EVPA angle, length ∝ |P|, gated by `pcut` / `mcut`. With `vec_cfun=True` the tick midpoints are sampled by |m| via a small side colorbar.


In [12]:
# Quick polarized image so the ticks have something to render. Build it
# locally (independent of im_dash above, which is Stokes-only).
im_pol = eh.image.make_empty(64, 200 * eh.RADPERUAS, 17.761, -29.0, rf=230e9)
im_pol = im_pol.add_gauss(1.0, (60 * eh.RADPERUAS, 60 * eh.RADPERUAS, 0, 0, 0))
im_pol.add_qu(0.20 * im_pol.imarr(), 0.10 * im_pol.imarr())
im_pol.add_v(0.02 * im_pol.imarr())

obs_pol = im_pol.observe(eht_dash, tint=30, tadv=600,
                          tstart=0, tstop=24, bw=4e9,
                          sgrscat=False, ampcal=False, phasecal=False)

fig_pol = interactive.dashboard(
    im_pol, obs_pol, ct_dash,
    plotp=True, nvec=18, pcut=0.05, mcut=0.01, vec_cfun=True,
    show=False,
)
interactive.display(fig_pol)


Generating empty observation file . . . 
Producing clean visibilities from image with nfft FT . . . 
Adding gain + phase errors to data and applying a priori calibration . . . 
   Applying gain corruption: ampcal-->False
   Applying atmospheric phase corruption: phasecal-->False
Adding thermal noise to data . . . 
Producing clean visibilities from image with direct FT . . . 


## `dashboard` data-product dropdown

Panel 2 hosts a dropdown — switch between amp / vis / phase / χ-residual / cphase / log-camp options without rebuilding the figure. The closure-quantity options (cphase, logcamp) now plot *every* triangle (or quadrangle) overlaid — each one a palette colour, the model rendered with open `x` markers and the same colour for each path. Cap the count with `n_triangles=` / `n_quadrangles=`, or filter to one with `triangle=` / `quadrangle=`.


In [13]:
# Enumerate the triangles + quadrangles the dashboard will plot in panel 2,
# sorted by sample count.
from ehtim.plotting.interactive import _enumerate_triangles, _enumerate_quads
tris = _enumerate_triangles(obs_dash, None, 12)
quads = _enumerate_quads(obs_dash, None, 12)
print(f'top {len(tris)} triangles: {tris[:5]} ...')
print(f'top {len(quads)} quadrangles: {quads[:5]} ...')

# Render the dashboard defaulting to the closure-phase panel so the
# multi-triangle overlay is visible right away.
fig_dp = interactive.dashboard(
    im_dash, obs_dash, ct_dash,
    default_product='cphase_vs_time',
    show=False,
)
interactive.display(fig_dp)


top 12 triangles: [('ALMA', 'APEX', 'SPT'), ('JCMT', 'SMA', 'SPT'), ('ALMA', 'APEX', 'LMT'), ('ALMA', 'LMT', 'SPT'), ('APEX', 'LMT', 'SPT')] ...
top 12 quadrangles: [('ALMA', 'APEX', 'LMT', 'SPT'), ('ALMA', 'APEX', 'LMT', 'SMT'), ('ALMA', 'LMT', 'SMT', 'SPT'), ('APEX', 'LMT', 'SMT', 'SPT'), ('ALMA', 'APEX', 'SMT', 'SPT')] ...
Producing clean visibilities from image with direct FT . . . 


In [15]:
# Filtering to a single triangle/quadrangle via kwargs (the user-facing way).
# Pass site names; if that closure path has no data the panel is simply empty.
interactive.dashboard(
    im_dash, obs_dash, ct_dash,
    triangle=('ALMA', 'LMT', 'SMA'),           # inspect this specific triangle
    quadrangle=('ALMA', 'LMT', 'SMA', 'SMT'),  # inspect this specific quadrangle
    default_product='cphase_vs_time',
    show=False,
)


Producing clean visibilities from image with direct FT . . . 


## `obs_helpers.uv_area_triangle` / `uv_area_quadrangle`

These power the new "vs triangle / quadrangle area" data-product panels and are also available standalone for the matplotlib plotters.

- Triangle uv-area: `A = (1/2) · |u1·v2 − u2·v1|` from two of its three baseline vectors.
- Quadrangle uv-area: sum of the two triangles formed by three independent baseline vectors.


In [16]:
from ehtim.observing import obs_helpers as obsh
import numpy as np

# Right triangle with legs along u and v: area = 1/2 * base * height = 1.
assert obsh.uv_area_triangle(1.0, 0.0, 0.0, 2.0) == 1.0

# Vectorised over multiple baselines (matches the cphase_tri / camp_quad shape).
u1, v1 = np.array([1.0, 2.0]), np.array([0.0, 0.0])
u2, v2 = np.array([0.0, 0.0]), np.array([2.0, 3.0])
print('triangle areas:', obsh.uv_area_triangle(u1, v1, u2, v2))

# Unit square with corners (0,0), (1,0), (1,1), (0,1): baselines (1,0), (0,1), (-1,0).
print('unit square uv-area:', obsh.uv_area_quadrangle(1.0, 0.0, 0.0, 1.0, -1.0, 0.0))


triangle areas: [1. 3.]
unit square uv-area: 1.0
